# Q2. 特征值的几何意义

给定矩阵 $A$：

$$A = \begin{bmatrix} 3 & 1 \\ 0 & 2 \end{bmatrix}$$

手算特征值和特征向量，然后解释：为什么 PCA（主成分分析）的核心就是对协方差矩阵做特征值分解？用一句话说清楚特征向量代表什么方向。

## 概念速览

### 特征值（Eigenvalue）与特征向量（Eigenvector）

对于方阵 $A$，如果存在非零向量 $v$ 使得：

$$A v = \lambda v$$

则称 $\lambda$ 为**特征值**，$v$ 为对应的**特征向量**。

**几何直觉**：矩阵 $A$ 作用于空间中的大多数向量时，方向会改变；而特征向量是那些**方向不变、只被拉伸/压缩的特殊方向**。特征值就是拉伸的倍数。

---

### 协方差矩阵（Covariance Matrix）

对于数据矩阵 $X \in \mathbb{R}^{n \times d}$（$n$ 个样本，$d$ 个特征，已中心化），协方差矩阵定义为：

$$\Sigma = \frac{1}{n-1} X^T X \in \mathbb{R}^{d \times d}$$

- 对角线元素 $\Sigma_{ii}$：第 $i$ 个特征的**方差**（自己和自己的相关性）
- 非对角线元素 $\Sigma_{ij}$：第 $i$ 和第 $j$ 个特征之间的**协方差**（两个维度一起变化的程度）
- $\Sigma$ 是**对称半正定矩阵**，因此特征值均 $\geq 0$

---

### PCA（主成分分析）

PCA 是一种**降维**方法，目标是找到一组新的正交坐标轴，使得数据在这些轴上的**方差最大**（即保留最多信息）。

步骤：
1. 对数据中心化
2. 计算协方差矩阵 $\Sigma$
3. 对 $\Sigma$ 做特征值分解：$\Sigma = Q \Lambda Q^T$
4. 按特征值从大到小排序，取前 $k$ 个特征向量作为新坐标轴
5. 将数据投影到这 $k$ 个方向上

**一句话**：协方差矩阵的特征向量代表数据**方差最大的方向**，特征值代表该方向上的**方差大小**。

> 为什么 PCA 的核心是特征值分解？因为我们要找的「让方差最大的方向」，数学上等价于求协方差矩阵的特征向量——这是一个约束优化问题（Lagrange 乘数法），其解恰好就是特征向量，最大方差恰好就是最大特征值。

## 为什么 PCA 是对协方差矩阵找特征向量？

这是整个 PCA 最核心的问题，答案来自一个优化推导。

**我们真正想解决的问题**

找一个方向（单位向量）$w$，使数据投影到这个方向上的**方差最大**。

投影后的方差是：

$$\text{Var}(Xw) = w^T \Sigma w$$

所以问题变成一个约束优化：

$$\max_w \ w^T \Sigma w \quad \text{subject to} \quad \|w\| = 1$$

**用 Lagrange 乘数法求解**

构造 Lagrangian：

$$L = w^T \Sigma w - \lambda (w^T w - 1)$$

对 $w$ 求导并令其为零：

$$\frac{\partial L}{\partial w} = 2\Sigma w - 2\lambda w = 0 \implies \Sigma w = \lambda w$$

**这正好就是特征向量的定义。**

**结论**

- 让投影方差最大的方向 $w$，必须是 $\Sigma$ 的特征向量
- 此时最大方差 $= w^T \Sigma w = w^T (\lambda w) = \lambda$，就是对应的特征值
- 所以**特征值越大 = 该方向上方差越大 = 信息越多**

取最大的 $k$ 个特征值对应的特征向量，就是保留方差最多的 $k$ 个方向，这就是 PCA。

## 手算特征值与特征向量

$$A = \begin{bmatrix} 3 & 1 \\ 0 & 2 \end{bmatrix}$$

### Step 1：求特征值

解特征方程 $\det(A - \lambda I) = 0$：

$$\det \begin{bmatrix} 3-\lambda & 1 \\ 0 & 2-\lambda \end{bmatrix} = (3-\lambda)(2-\lambda) - 0 = 0$$

$$\lambda^2 - 5\lambda + 6 = 0 \implies (\lambda-3)(\lambda-2) = 0$$

$$\boxed{\lambda_1 = 3, \quad \lambda_2 = 2}$$

### Step 2：求特征向量

**对 $\lambda_1 = 3$**，解 $(A - 3I)v = 0$：

$$\begin{bmatrix} 0 & 1 \\ 0 & -1 \end{bmatrix} v = 0 \implies v_2 = 0 \implies \boxed{v_1 = \begin{bmatrix} 1 \\ 0 \end{bmatrix}}$$

**对 $\lambda_2 = 2$**，解 $(A - 2I)v = 0$：

$$\begin{bmatrix} 1 & 1 \\ 0 & 0 \end{bmatrix} v = 0 \implies v_1 + v_2 = 0 \implies \boxed{v_2 = \begin{bmatrix} 1 \\ -1 \end{bmatrix}}$$

**几何解释**：$v_1 = [1, 0]^T$ 方向被拉伸 3 倍，$v_2 = [1, -1]^T$ 方向被拉伸 2 倍，其他方向会发生旋转。

In [1]:
import torch
import numpy as np

A = torch.tensor([[3.0, 1.0],
                  [0.0, 2.0]])

# torch.linalg.eig 返回复数，对实对称矩阵用 eigh 更合适
# A 不是对称矩阵，用 eig
eigenvalues, eigenvectors = torch.linalg.eig(A)

print("特征值:", eigenvalues.real)
print("特征向量（每列为一个特征向量）:")
print(eigenvectors.real)

# 验证 Av = λv
for i in range(2):
    lam = eigenvalues[i].real
    v   = eigenvectors[:, i].real
    Av  = A @ v
    lv  = lam * v
    print(f"\nλ={lam:.1f}: Av={Av.tolist()},  λv={lv.tolist()},  一致={torch.allclose(Av, lv, atol=1e-5)}")

特征值: tensor([3., 2.])
特征向量（每列为一个特征向量）:
tensor([[ 1.0000, -0.7071],
        [ 0.0000,  0.7071]])

λ=3.0: Av=[3.0, 0.0],  λv=[3.0, 0.0],  一致=True

λ=2.0: Av=[-1.4142134189605713, 1.4142135381698608],  λv=[-1.4142135381698608, 1.4142135381698608],  一致=True


In [ ]:
import torch

X = torch.tensor([[1., 3.],
                  [2., 5.],
                  [3., 4.],
                  [4., 2.]])

# Step 1: center
X_c = X - X.mean(dim=0)
print("Centered X:\n", X_c)

# Step 2 & 3: covariance matrix
cov = X_c.T @ X_c / (len(X) - 1)
print("\nCovariance matrix:\n", cov)
# diagonal = variance of each feature
# off-diagonal = negative → features move in opposite directions

## 具体例子：手算协方差矩阵（4×2 矩阵）

原始数据（4 个样本，2 个特征）：

$$X = \begin{bmatrix} 1 & 3 \\ 2 & 5 \\ 3 & 4 \\ 4 & 2 \end{bmatrix}$$

**Step 1：中心化**，各列减去均值（$\bar{x}_1 = 2.5,\ \bar{x}_2 = 3.5$）：

$$X_c = \begin{bmatrix} -1.5 & -0.5 \\ -0.5 & 1.5 \\ 0.5 & 0.5 \\ 1.5 & -1.5 \end{bmatrix}$$

**Step 2：计算 $X_c^T X_c$**：

$$X_c^T X_c = \begin{bmatrix} (-1.5)^2+(-0.5)^2+0.5^2+1.5^2 & (-1.5)(-0.5)+(-0.5)(1.5)+(0.5)(0.5)+(1.5)(-1.5) \\ \cdots & (-0.5)^2+1.5^2+0.5^2+(-1.5)^2 \end{bmatrix} = \begin{bmatrix} 5 & -2 \\ -2 & 5 \end{bmatrix}$$

**Step 3：除以 $n - 1 = 3$**：

$$\Sigma = \frac{1}{3}\begin{bmatrix} 5 & -2 \\ -2 & 5 \end{bmatrix} \approx \begin{bmatrix} 1.67 & -0.67 \\ -0.67 & 1.67 \end{bmatrix}$$

**如何读这个矩阵：**
- 对角线 $1.67$：两个特征各自的方差（变化幅度差不多）
- 非对角线 $-0.67$：**负协方差**，说明特征 1 大时特征 2 倾向于小（看原始数据：$x_1=4$ 时 $x_2=2$，反向变化）

## 具体例子：PCA 从 4 维降到 2 维

你有 $n$ 个样本，每个样本有 4 个特征，数据矩阵 $X \in \mathbb{R}^{n \times 4}$。  
目标：找到 $W \in \mathbb{R}^{4 \times 2}$，使得 $XW \in \mathbb{R}^{n \times 2}$。

| 步骤 | 操作 | Shape |
|------|------|-------|
| 中心化 | $X \leftarrow X - \bar{X}$ | $n \times 4$ |
| 计算协方差矩阵 | $\Sigma = \frac{1}{n-1}X^TX$ | $4 \times 4$ |
| 特征值分解 | $\Sigma = Q\Lambda Q^T$，按 $\lambda$ 从大到小排序 | 4 个长度为 4 的特征向量 |
| 取前 2 个特征向量 | $W = [v_1\ v_2]$ | $4 \times 2$ |
| 投影 | $X_{\text{reduced}} = XW$ | $n \times 2$ |

**为什么选前 2 个？** 特征值的大小等于数据在该方向上的方差。假设 4 个特征值为 $[9.2,\ 3.1,\ 0.4,\ 0.1]$，前两个方向已经解释了 $\frac{9.2+3.1}{12.8} \approx 96\%$ 的总方差，几乎没有信息损失。

In [ ]:
import torch

torch.manual_seed(0)

n = 100
X = torch.randn(n, 4)                        # n samples, 4 features

# Step 1: center
X = X - X.mean(dim=0)                        # (n, 4)

# Step 2: covariance matrix
cov = X.T @ X / (n - 1)                      # (4, 4)

# Step 3: eigendecomposition (eigh for symmetric matrices — guaranteed real, sorted ascending)
eigenvalues, eigenvectors = torch.linalg.eigh(cov)   # eigenvalues: (4,), eigenvectors: (4,4)

# Sort descending
idx = eigenvalues.argsort(descending=True)
eigenvalues  = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]          # each column is an eigenvector

print("Eigenvalues:", eigenvalues.tolist())
variance_explained = eigenvalues / eigenvalues.sum()
print("Variance explained:", [f"{v:.1%}" for v in variance_explained.tolist()])
print(f"Top 2 capture: {variance_explained[:2].sum():.1%} of total variance")

# Step 4: projection matrix W = top 2 eigenvectors
W = eigenvectors[:, :2]                      # (4, 2)
print("\nW shape:", W.shape)                  # (4, 2)

# Step 5: project
X_reduced = X @ W                            # (n, 2)
print("X_reduced shape:", X_reduced.shape)   # (n, 2)